# HR Data Dashboard

Interactive visualizations for exploring generated HR data using Plotly.

**Requirements:**
```bash
pip install plotly
# or
pip install hr-data-generator[viz]
```

## Setup

In [ ]:
from hr_data_generator import generate_hr_data
import pandas as pd
import plotly.express as px
import plotly.io as pio
from datetime import datetime

# Set default template for consistent styling
pio.templates.default = "plotly_white"

# Display settings
pd.set_option('display.max_columns', None)

In [ ]:
def get_current_records(df, date_col='end_date'):
    """Filter to current (active) records where end_date is null."""
    return df[df[date_col].isna()].copy()

## Generate Data

In [ ]:
# Generate HR dataset
data = generate_hr_data(n_employees=500, seed=42)

# Extract tables
employees = data['employee']
job_assignments = data['employee_job_assignment']
org_assignments = data['employee_org_assignment']
compensation = data['employee_compensation']
performance = data['employee_performance']
locations = data['location']
job_roles = data['job_role']

# Get current records for time-variant tables
current_jobs = get_current_records(job_assignments)
current_orgs = get_current_records(org_assignments)
current_comp = get_current_records(compensation)

print(f"Generated {len(employees)} employees")
print(f"Tables: {list(data.keys())}")

---
## 1. Organizational Overview

### Workforce by Business Unit

In [ ]:
# Business unit distribution
bu_counts = current_orgs['business_unit'].value_counts().reset_index()
bu_counts.columns = ['Business Unit', 'Headcount']
bu_counts['Percentage'] = (bu_counts['Headcount'] / bu_counts['Headcount'].sum() * 100).round(1)

fig = px.bar(
    bu_counts,
    x='Business Unit',
    y='Headcount',
    text='Headcount',
    title='Workforce by Business Unit',
    hover_data=['Percentage'],
    color='Headcount',
    color_continuous_scale='Blues'
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, coloraxis_showscale=False)
fig.show()

### Seniority Level Distribution

In [ ]:
# Seniority level distribution
seniority_counts = current_jobs['seniority_level'].value_counts().sort_index().reset_index()
seniority_counts.columns = ['Seniority Level', 'Count']
seniority_counts['Percentage'] = (seniority_counts['Count'] / seniority_counts['Count'].sum() * 100).round(1)

# Map seniority levels to labels
seniority_labels = {1: '1 - Entry', 2: '2 - Junior', 3: '3 - Mid', 4: '4 - Senior', 5: '5 - Lead', 6: '6 - Executive'}
seniority_counts['Level Label'] = seniority_counts['Seniority Level'].map(seniority_labels)

fig = px.bar(
    seniority_counts,
    x='Level Label',
    y='Count',
    text='Count',
    title='Seniority Level Distribution',
    hover_data=['Percentage'],
    color='Count',
    color_continuous_scale='Greens'
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, coloraxis_showscale=False, xaxis_title='Seniority Level')
fig.show()

---
## 2. Demographics

### Gender Breakdown

In [ ]:
# Gender distribution pie chart
gender_counts = employees['gender'].value_counts().reset_index()
gender_counts.columns = ['Gender', 'Count']

fig = px.pie(
    gender_counts,
    values='Count',
    names='Gender',
    title='Gender Breakdown',
    color='Gender',
    color_discrete_map={'Male': '#636EFA', 'Female': '#EF553B'}
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

### Tenure Distribution

In [ ]:
# Calculate tenure in years
employees_tenure = employees.copy()
employees_tenure['hire_date'] = pd.to_datetime(employees_tenure['hire_date'])
employees_tenure['tenure_years'] = ((datetime.now() - employees_tenure['hire_date']).dt.days / 365.25).round(1)

fig = px.histogram(
    employees_tenure,
    x='tenure_years',
    nbins=20,
    title='Tenure Distribution',
    labels={'tenure_years': 'Tenure (Years)', 'count': 'Number of Employees'},
    color_discrete_sequence=['#00CC96']
)
fig.update_layout(bargap=0.1)
fig.show()

---
## 3. Compensation & Performance

### Salary by Seniority Level

In [ ]:
# Join compensation with job assignments to get seniority level
comp_with_seniority = current_comp.merge(
    current_jobs[['employee_id', 'seniority_level', 'job_title']],
    on='employee_id',
    how='left'
)

# Map seniority levels to labels
comp_with_seniority['Seniority'] = comp_with_seniority['seniority_level'].map(seniority_labels)

fig = px.box(
    comp_with_seniority,
    x='Seniority',
    y='base_salary',
    title='Salary Distribution by Seniority Level',
    labels={'base_salary': 'Base Salary', 'Seniority': 'Seniority Level'},
    color='Seniority',
    hover_data=['job_title']
)
fig.update_layout(showlegend=False)
fig.show()

### Performance Rating Distribution

In [ ]:
# Performance rating distribution
perf_counts = performance.groupby(['rating', 'rating_label']).size().reset_index(name='Count')
perf_counts = perf_counts.sort_values('rating')
perf_counts['Percentage'] = (perf_counts['Count'] / perf_counts['Count'].sum() * 100).round(1)

# Color mapping for ratings
rating_colors = {
    'Exceptional': '#00CC96',
    'Exceeds Expectations': '#19D3F3',
    'Meets Expectations': '#636EFA',
    'Needs Improvement': '#FFA15A',
    'Unsatisfactory': '#EF553B'
}

fig = px.bar(
    perf_counts,
    x='rating_label',
    y='Count',
    text='Count',
    title='Performance Rating Distribution',
    hover_data=['Percentage'],
    color='rating_label',
    color_discrete_map=rating_colors
)
fig.update_traces(textposition='outside')
fig.update_layout(xaxis_title='Rating', showlegend=False)
fig.show()

---
## 4. Geographic & Job Structure

### Top 10 Locations by Headcount

In [ ]:
# Top 10 locations
location_counts = employees.merge(
    locations[['location_id', 'city', 'country']],
    on='location_id',
    how='left'
)
location_summary = location_counts.groupby(['city', 'country']).size().reset_index(name='Headcount')
location_summary = location_summary.sort_values('Headcount', ascending=False).head(10)
location_summary['Location'] = location_summary['city'] + ', ' + location_summary['country']

fig = px.bar(
    location_summary,
    x='Headcount',
    y='Location',
    orientation='h',
    title='Top 10 Locations by Headcount',
    text='Headcount',
    color='Headcount',
    color_continuous_scale='Purples'
)
fig.update_traces(textposition='outside')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, coloraxis_showscale=False)
fig.show()

### Job Level by Job Family

In [ ]:
# Job level distribution by job family (stacked bar)
jobs_with_family = current_jobs.merge(
    job_roles[['job_id', 'job_family', 'job_level']],
    on='job_id',
    how='left',
    suffixes=('', '_role')
)

job_family_level = jobs_with_family.groupby(['job_family', 'job_level']).size().reset_index(name='Count')

fig = px.bar(
    job_family_level,
    x='job_family',
    y='Count',
    color='job_level',
    title='Job Level Distribution by Job Family',
    labels={'job_family': 'Job Family', 'job_level': 'Job Level'},
    barmode='stack',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(xaxis_title='Job Family', legend_title='Job Level')
fig.show()

---
## 5. Career Progression

### Salary vs Tenure

In [ ]:
# Salary vs Tenure scatter plot
scatter_data = employees_tenure.merge(
    current_comp[['employee_id', 'base_salary']],
    on='employee_id',
    how='left'
).merge(
    current_jobs[['employee_id', 'seniority_level', 'job_title']],
    on='employee_id',
    how='left'
)

scatter_data['Seniority'] = scatter_data['seniority_level'].map(seniority_labels)

fig = px.scatter(
    scatter_data,
    x='tenure_years',
    y='base_salary',
    color='Seniority',
    title='Salary vs Tenure (by Seniority Level)',
    labels={
        'tenure_years': 'Tenure (Years)',
        'base_salary': 'Base Salary',
        'Seniority': 'Seniority Level'
    },
    hover_data=['first_name', 'last_name', 'job_title'],
    opacity=0.7
)
fig.update_layout(legend_title='Seniority Level')
fig.show()

---
## Summary Statistics

In [ ]:
# Key metrics summary
summary = {
    'Total Employees': len(employees),
    'Business Units': current_orgs['business_unit'].nunique(),
    'Unique Locations': employees['location_id'].nunique(),
    'Job Roles': current_jobs['job_id'].nunique(),
    'Avg Tenure (Years)': round(employees_tenure['tenure_years'].mean(), 1),
    'Avg Salary': f"${current_comp['base_salary'].mean():,.0f}",
    'Median Salary': f"${current_comp['base_salary'].median():,.0f}",
    'Performance Reviews': len(performance)
}

summary_df = pd.DataFrame(list(summary.items()), columns=['Metric', 'Value'])
print("\n" + "="*40)
print("       HR DATA SUMMARY")
print("="*40)
for _, row in summary_df.iterrows():
    print(f"{row['Metric']:.<25} {row['Value']}")
print("="*40)